# LH 장기공실 분석 코드 (반입용 노트북)

1. 위에서부터 **아래 '파일 만들기' 셀들을 모두 실행**하면 이 노트북과 같은 폴더에 분석 코드 파일 12개가 생성됩니다.
2. `config.py` 셀에서 파일명·컬럼명을 실제 데이터에 맞게 고친 뒤 다시 실행하세요.
3. 마지막 셀에서 데이터 폴더 경로를 넣고 실행합니다.

In [ ]:
%%writefile config.py
"""LH 임대주택 공실 분석 - 공통 설정.

센터에서는 DATA_DIR 만 실제 데이터 폴더로 바꾸면 된다.
"""
import os

# 데이터 위치: 환경변수 LH_DATA_DIR 가 있으면 그것을, 없으면 ../fake_data 를 쓴다.
DATA_DIR = os.environ.get("LH_DATA_DIR", os.path.join(os.path.dirname(__file__), "..", "fake_data"))
OUT_DIR = os.environ.get("LH_OUT_DIR", os.path.join(os.path.dirname(__file__), "..", "output"))

FILES = {
    "contract": "전국 건설임대주택 계약 기초데이터(계약).csv",
    "fee": "전국 건설임대주택 계약 기초데이터(관리비).csv",
    "kcb": "TB_KCB_INCOME_STAT_DATASET.csv",
    "skt_age": "seoul_flow_age.csv",
}

# 데이터 마지막 시점 (정의서 기준). 이 시점까지 다음 계약이 없으면 '공실 진행 중'으로 본다.
DATA_END_YM = 202607

# 반출 심의용: 이 건수 미만인 집계 칸은 가린다(센터 기준에 맞게 조정).
MIN_CELL = 10

# 장기 공실 기준(개월)
LONG_VACANCY = [6, 12, 24]

# 실제 파일 컬럼명(영문) -> 분석용 한글명. 파일이 이미 한글 컬럼이면 그대로 둔다.
CONTRACT_COLS = {
    "CNP_NM": "시도", "SGG_NM": "시군구", "LGDN_CD": "법정동코드", "HO_ADM_NO": "호관리번호",
    "SPL_TP_NM": "공급유형", "LS_BLD_DS_NM": "주택유형", "HTN_FMLA_DS_NM": "난방방식",
    "CCW_DT": "준공년월", "DDO_AR": "전용면적", "RM_CNT": "방수", "FST_CTRT_DT": "최초계약년월",
    "MVIN_DT": "입주년월", "LS_ST_DT": "임대시작년월", "LS_ED_DT": "임대종료년월",
    "CNCT_DT": "해약년월", "LS_GMY": "임대보증금", "RFE": "임대료",
}
FEE_COLS = {
    "HO_ADM_NO": "호관리번호", "STND_YM": "기준년월", "TMM_ADM_XPS": "당월관리비",
    "HTN_XPS": "난방비", "GAS_UFE": "가스사용료", "ELEC_CHRG": "전기요금", "WRA_CHRG": "수도료",
}
KCB_COLS = {"BS_YR_QT": "기준시점", "RES_COM_CD": "거주기준", "EMD_CD": "행정동코드",
            "SEX_CD": "성", "AGE_CD": "연령"}

# 면적 구간(㎡)
AREA_BINS = [0, 30, 40, 50, 60, 85, 1000]
AREA_LABELS = ["30미만", "30~40", "40~50", "50~60", "60~85", "85이상"]


In [ ]:
%%writefile common.py
"""공통 함수: 읽기, 날짜 변환, 반출용 마스킹/저장."""
import os
import numpy as np
import pandas as pd
import config


def ensure_out(sub=""):
    path = os.path.join(config.OUT_DIR, sub)
    os.makedirs(path, exist_ok=True)
    return path


def read_csv(key, usecols=None, sep=",", chunksize=None):
    """정의서 기준 utf-8. 깨지면 cp949로 다시 시도."""
    path = os.path.join(config.DATA_DIR, config.FILES[key])
    for enc in ("utf-8", "utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, dtype=str, usecols=usecols,
                               chunksize=chunksize, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError("인코딩을 확인하세요: " + path)


def rename(df, mapping):
    return df.rename(columns={k: v for k, v in mapping.items() if k in df.columns})


def ym_to_idx(s):
    """'YYYYMM' 문자열 -> 월 번호(년*12+월-1). 비었거나 이상하면 NaN."""
    s = pd.Series(s).astype(str).str.replace(r"\D", "", regex=True).str[:6]
    y = pd.to_numeric(s.str[:4], errors="coerce")
    m = pd.to_numeric(s.str[4:6], errors="coerce")
    idx = y * 12 + (m - 1)
    idx[(m < 1) | (m > 12) | (y < 1950) | (y > 2100)] = np.nan
    return idx


def idx_to_year(idx):
    return (idx // 12).astype("Int64")


def mask_small(df, count_col="호수", min_cell=None):
    """건수가 기준 미만인 행의 값은 가린다(반출 심의 대비)."""
    min_cell = min_cell or config.MIN_CELL
    out = df.copy()
    small = out[count_col] < min_cell
    for c in out.columns:
        if c != count_col and pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].astype(float)
            out.loc[small, c] = np.nan
    out[count_col] = out[count_col].astype(object)
    out.loc[small, count_col] = "<%d" % min_cell
    return out


def save(df, name, sub="export"):
    path = os.path.join(ensure_out(sub), name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("저장:", path, df.shape)
    return path


def setup_korean_font():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if any(name in f.name for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False
    return plt


In [ ]:
%%writefile s01profile.py
"""[1차 방문] 데이터 요약표 만들기 → 반출 신청용.

개인/호 단위 값은 내보내지 않고 컬럼별 통계만 만든다.
결과: output/export/01_컬럼요약_*.csv, 01_분포_*.csv, 01_환경정보.csv
"""
import platform
import sys
import pandas as pd
import common
import config


def profile(df, name):
    rows = []
    for c in df.columns:
        s = df[c]
        num = pd.to_numeric(s, errors="coerce")
        is_num = num.notna().mean() > 0.9
        rows.append({
            "컬럼": c, "행수": len(s), "결측률(%)": round(100 * (s.isna() | (s.astype(str).str.strip() == "")).mean(), 2),
            "고유값수": s.nunique(dropna=True),
            "최소": num.min() if is_num else "", "최대": num.max() if is_num else "",
            "중앙값": num.median() if is_num else "",
        })
    out = pd.DataFrame(rows)
    # 식별자 컬럼은 최소/최대값을 지운다
    out[["최소", "최대", "중앙값"]] = out[["최소", "최대", "중앙값"]].astype(object)
    out.loc[out["컬럼"].isin(["호관리번호", "법정동코드", "행정동코드"]), ["최소", "최대", "중앙값"]] = ""
    common.save(out, "01_컬럼요약_%s.csv" % name)


def dist(df, cols, name):
    for c in cols:
        if c in df.columns:
            t = df[c].fillna("(결측)").value_counts().rename_axis(c).reset_index(name="호수")
            common.save(common.mask_small(t), "01_분포_%s_%s.csv" % (name, c))


con = common.rename(common.read_csv("contract"), config.CONTRACT_COLS)
profile(con, "계약")
dist(con, ["시도", "공급유형", "주택유형", "난방방식", "방수"], "계약")
# 연도별 계약 시작 건수
y = (common.ym_to_idx(con["임대시작년월"]) // 12).value_counts().sort_index()
common.save(common.mask_small(y.rename_axis("임대시작연도").reset_index(name="호수")), "01_분포_계약_임대시작연도.csv")
print("호 수:", con["호관리번호"].nunique(), "/ 호당 평균 계약 수:", round(len(con) / con["호관리번호"].nunique(), 2))

try:
    fee = common.rename(next(iter(common.read_csv("fee", chunksize=1000000))), config.FEE_COLS)  # 대용량: 앞 100만 행만 요약
    profile(fee, "관리비")
except FileNotFoundError:
    print("관리비 파일 없음 - 건너뜀")
try:
    kcb = common.rename(common.read_csv("kcb"), config.KCB_COLS)
    profile(kcb, "KCB")
except FileNotFoundError:
    print("KCB 파일 없음 - 건너뜀")

# 분석 PC 환경 정보
lines = ["python " + sys.version, "platform " + platform.platform()]
for m in ["pandas", "numpy", "matplotlib", "sklearn", "scipy", "statsmodels", "lifelines", "geopandas"]:
    try:
        mod = __import__(m)
        lines.append("%s %s" % (m, getattr(mod, "__version__", "?")))
    except ImportError:
        lines.append("%s (없음)" % m)
common.save(pd.DataFrame({"항목": [l.split(" ", 1)[0] for l in lines], "버전": [l.split(" ", 1)[1] for l in lines]}),
            "01_환경정보.csv")
print("\n".join(lines))


In [ ]:
%%writefile s02vacancy.py
"""[본 분석 1] 호별 계약 이력으로 공실 구간 만들기.

공실 정의
- 같은 호에서 한 계약이 끝난 뒤(해약년월이 있으면 해약년월, 없으면 임대종료년월)
  다음 계약 임대시작년월까지의 빈 기간(개월).
- 다음 계약이 없고 끝난 시점이 데이터 마지막 시점(2026.07) 이전이면 '공실 진행 중'(관측 중단).
- 준공 후 첫 임대시작까지 = '초기 미임대 기간'.
결과(센터 안에서만 사용, 반출 금지): output/work/공실구간.parquet|csv, 호특성.csv
"""
import numpy as np
import pandas as pd
import common
import config

con = common.rename(common.read_csv("contract"), config.CONTRACT_COLS)
for c in ["준공년월", "임대시작년월", "임대종료년월", "해약년월", "최초계약년월", "입주년월"]:
    con[c + "_i"] = common.ym_to_idx(con[c])
for c in ["전용면적", "방수", "임대보증금", "임대료"]:
    con[c] = pd.to_numeric(con[c], errors="coerce")
end_data = common.ym_to_idx([str(config.DATA_END_YM)])[0]

con = con.dropna(subset=["호관리번호", "임대시작년월_i"]).sort_values(["호관리번호", "임대시작년월_i"])
con["종료_i"] = con["해약년월_i"].fillna(con["임대종료년월_i"])
con["중도해약"] = con["해약년월_i"].notna() & (con["해약년월_i"] < con["임대종료년월_i"])
con["다음시작_i"] = con.groupby("호관리번호")["임대시작년월_i"].shift(-1)

# 공실 구간: 계약 종료 다음 달부터 다음 계약 시작 전 달까지
gap = con["다음시작_i"] - con["종료_i"] - 1
ongoing = con["다음시작_i"].isna() & (con["종료_i"] < end_data)
ep = con.loc[(gap > 0) | ongoing].copy()
ep["공실개월"] = np.where(ep["다음시작_i"].notna(), ep["다음시작_i"] - ep["종료_i"] - 1, end_data - ep["종료_i"])
ep["진행중"] = ep["다음시작_i"].isna()
ep["공실시작연도"] = ((ep["종료_i"] + 1) // 12).astype(int)
ep["직전계약_중도해약"] = ep["중도해약"]
keep = ["호관리번호", "시도", "시군구", "법정동코드", "공급유형", "주택유형", "난방방식", "전용면적", "방수",
        "준공년월_i", "임대보증금", "임대료", "공실시작연도", "공실개월", "진행중", "직전계약_중도해약"]
ep = ep[keep]

# 호 특성(가장 최근 계약 기준) + 공실 경험 요약
last = con.groupby("호관리번호").tail(1).set_index("호관리번호")
unit = last[["시도", "시군구", "법정동코드", "공급유형", "주택유형", "난방방식", "전용면적", "방수",
             "준공년월_i", "임대보증금", "임대료"]].copy()
first_start = con.groupby("호관리번호")["임대시작년월_i"].min()
unit["초기미임대개월"] = (first_start - unit["준공년월_i"]).clip(lower=0)
unit["계약수"] = con.groupby("호관리번호").size()
unit["중도해약수"] = con.groupby("호관리번호")["중도해약"].sum()
g = ep.groupby("호관리번호")["공실개월"]
unit["공실횟수"] = g.size()
unit["누적공실개월"] = g.sum()
unit["최장공실개월"] = g.max()
unit = unit.fillna({"공실횟수": 0, "누적공실개월": 0, "최장공실개월": 0})
unit["현재공실"] = ep.groupby("호관리번호")["진행중"].any().reindex(unit.index).fillna(False)
unit["현재공실개월"] = ep[ep["진행중"]].set_index("호관리번호")["공실개월"].reindex(unit.index)
unit["준공연차"] = ((end_data - unit["준공년월_i"]) / 12).round(1)
unit["면적구간"] = pd.cut(unit["전용면적"], config.AREA_BINS, labels=config.AREA_LABELS, right=False)
# 지역 내 임대료 상대수준(시군구 내 ㎡당 임대료 분위)
unit["㎡당임대료"] = unit["임대료"] / unit["전용면적"]
unit["임대료분위_시군구내"] = unit.groupby("시군구")["㎡당임대료"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 4, labels=["하", "중하", "중상", "상"]) if len(s) >= 4 else np.nan)

work = common.ensure_out("work")
ep.to_csv(work + "/공실구간.csv", index=False, encoding="utf-8-sig")
unit.reset_index().to_csv(work + "/호특성.csv", index=False, encoding="utf-8-sig")
print("공실 구간:", len(ep), "/ 호:", len(unit), "/ 현재 공실 호:", int(unit["현재공실"].sum()))
print("※ output/work 는 호 단위 자료라 반출하지 말 것")


In [ ]:
%%writefile s02bcomplex.py
"""[본 분석 1-b] '같은 단지로 보이는 집' 묶기 → 단지 단위 공실 집계.

LH 데이터에는 단지명이 없다. 대신 같은 법정동 + 같은 준공년월 + 같은 공급유형 + 같은 주택유형인
호들은 거의 같은 단지이므로 이를 '추정 단지'로 묶는다.
- 법정동코드가 시군구(5자리)까지만 들어 있으면 단지 구분이 거칠어지므로 경고를 출력한다.
결과
- output/work/추정단지.csv (센터 내부용)
- output/export/02b_추정단지별_공실.csv (반출용: 단지 키는 법정동코드·준공년월·유형만, 호 정보 없음, 소수 단지 가림)
"""
import pandas as pd
import common
import config

work = config.OUT_DIR + "/work"
unit = pd.read_csv(work + "/호특성.csv", dtype={"법정동코드": str})

code = unit["법정동코드"].fillna("").str.replace(r"\D", "", regex=True)
digits = code.str.rstrip("0").str.len().median()
n_dong = code.nunique()
print("법정동코드 고유값 %d개, 유효 자리수 중앙값 %.0f" % (n_dong, digits))
if digits <= 5:
    print("⚠ 법정동코드가 시군구 수준으로 보임 → 추정 단지 정확도 낮음. 조건 묶음(03) 분석을 우선 사용")

unit["준공년월"] = (unit["준공년월_i"] // 12).astype("Int64").astype(str) + \
                  ((unit["준공년월_i"] % 12 + 1).astype("Int64").astype(str).str.zfill(2))
key = ["시도", "시군구", "법정동코드", "준공년월", "공급유형", "주택유형"]
cx = unit.groupby(key, dropna=False).agg(
    호수=("호관리번호", "size"),
    현재공실호수=("현재공실", "sum"),
    현재공실_중위개월=("현재공실개월", "median"),
    평균누적공실개월=("누적공실개월", "mean"),
    최장공실_중위개월=("최장공실개월", "median"),
    평균전용면적=("전용면적", "mean"),
    소형비율=("전용면적", lambda s: (s < 40).mean()),
    중도해약_호당평균=("중도해약수", "mean"),
    초기미임대_중위개월=("초기미임대개월", "median"),
).reset_index()
cx["현재공실률(%)"] = 100 * cx["현재공실호수"] / cx["호수"]
cx["장기공실호비율(%)"] = unit.assign(L=(unit["최장공실개월"] >= 12) | (unit["현재공실개월"] >= 12)) \
    .groupby(key, dropna=False)["L"].mean().values * 100
cx = cx.round(2).sort_values("현재공실률(%)", ascending=False)
cx.insert(0, "추정단지ID", range(1, len(cx) + 1))
cx.to_csv(work + "/추정단지.csv", index=False, encoding="utf-8-sig")
print("추정 단지 수:", len(cx), "/ 단지당 평균 호수: %.1f" % cx["호수"].mean())
common.save(common.mask_small(cx), "02b_추정단지별_공실.csv")


In [ ]:
%%writefile s03aggregate.py
"""[본 분석 2] 반출용 집계표와 그래프.

모두 지역·유형 단위로 집계하고, 호수가 MIN_CELL 미만인 칸은 가린다.
결과: output/export/03_*.csv (반출은 CSV만 가능 → 그래프는 s05charts.py로 집에서 생성)
"""
import pandas as pd
import common
import config

work = config.OUT_DIR + "/work"
unit = pd.read_csv(work + "/호특성.csv", dtype={"법정동코드": str})
ep = pd.read_csv(work + "/공실구간.csv", dtype={"법정동코드": str})
L6, L12, L24 = config.LONG_VACANCY


def summarize(group_cols, name):
    u = unit.groupby(group_cols, observed=True).agg(
        호수=("호관리번호", "size"),
        현재공실호수=("현재공실", "sum"),
        공실경험호비율=("공실횟수", lambda s: round(100 * (s > 0).mean(), 1)),
        평균누적공실개월=("누적공실개월", "mean"),
        초기미임대_중위개월=("초기미임대개월", "median"),
        중도해약_호당평균=("중도해약수", "mean"),
    )
    cur = unit[unit["현재공실"]].groupby(group_cols, observed=True)["현재공실개월"]
    u["현재공실_중위개월"] = cur.median()
    u["현재공실_%d개월이상" % L12] = unit[unit["현재공실개월"] >= L12].groupby(group_cols, observed=True).size()
    e = ep.groupby(group_cols, observed=True)["공실개월"]
    u["공실구간수"] = e.size()
    u["공실_중위개월"] = e.median()
    for k in (L6, L12, L24):
        u["공실_%d개월이상비율" % k] = ep.assign(f=ep["공실개월"] >= k).groupby(group_cols, observed=True)["f"].mean() * 100
    u["현재공실률(%)"] = 100 * u["현재공실호수"] / u["호수"]
    u = u.round(2).reset_index().sort_values("호수", ascending=False)
    common.save(common.mask_small(u), "03_%s.csv" % name)
    return u


ep = ep.merge(unit[["호관리번호", "면적구간", "임대료분위_시군구내"]], on="호관리번호", how="left")
unit["준공연차구간"] = pd.cut(unit["준공연차"], [0, 5, 10, 15, 20, 100], labels=["5년미만", "5~10", "10~15", "15~20", "20년이상"])
ep = ep.merge(unit[["호관리번호", "준공연차구간"]], on="호관리번호", how="left")

summarize(["시도"], "시도별")
sgg = summarize(["시도", "시군구"], "시군구별")
summarize(["공급유형"], "공급유형별")
summarize(["면적구간"], "면적구간별")
summarize(["공급유형", "면적구간"], "공급유형x면적")
summarize(["방수"], "방수별")
summarize(["준공연차구간"], "준공연차별")
summarize(["임대료분위_시군구내"], "임대료분위별")
summarize(["주택유형"], "주택유형별")

# 연도별 공실 발생 추이
t = ep.groupby("공실시작연도").agg(호수=("호관리번호", "size"), 공실_중위개월=("공실개월", "median")).reset_index()
common.save(common.mask_small(t), "03_연도별_공실발생.csv")

print("집계표 저장 완료 (그래프는 집에서 s05charts.py 로 생성)")


In [ ]:
%%writefile s04model.py
"""[본 분석 3] 지역 소득(KCB)·관리비 결합 + 장기공실 위험 모델.

- KCB 행정동 소득분포 -> 시군구(코드 앞 5자리) 단위로 합쳐 '저소득 비율', '20·30대 비율' 계산
- 관리비(2023.01~) -> 호별 ㎡당 평균 관리비
- 목표: 현재 12개월 이상 공실이거나 과거 12개월 이상 공실 경험이 있는 호(장기공실 호)
- 모델: 로지스틱 회귀 + 그래디언트 부스팅. 반출은 성능·변수중요도(집계)만.
"""
import numpy as np
import pandas as pd
import common
import config

work = config.OUT_DIR + "/work"
unit = pd.read_csv(work + "/호특성.csv", dtype={"법정동코드": str})
unit["시군구코드"] = unit["법정동코드"].str[:5]

# KCB: 시군구 단위 소득 지표
try:
    kcb = common.rename(common.read_csv("kcb"), config.KCB_COLS)
    cnt = [c for c in kcb.columns if c.startswith("C") and c.endswith("_CNT")]
    kcb[cnt] = kcb[cnt].apply(pd.to_numeric, errors="coerce").fillna(0)
    kcb = kcb[kcb["기준시점"] == kcb["기준시점"].max()]
    kcb["시군구코드"] = kcb["행정동코드"].str[:5]
    kcb["전체"] = kcb[cnt].sum(axis=1)
    kcb["저소득"] = kcb[["C1_CNT", "C2_CNT"]].sum(axis=1)  # 월 150만원 이하
    kcb["청년"] = np.where(kcb["연령"].isin(["20", "30"]), kcb["전체"], 0)
    reg = kcb.groupby("시군구코드")[["전체", "저소득", "청년"]].sum()
    reg["저소득비율"] = reg["저소득"] / reg["전체"]
    reg["2030비율"] = reg["청년"] / reg["전체"]
    unit = unit.merge(reg[["저소득비율", "2030비율"]], left_on="시군구코드", right_index=True, how="left")
    common.save(reg.reset_index()[["시군구코드", "저소득비율", "2030비율"]].round(4), "04_시군구_소득지표.csv")
except FileNotFoundError:
    print("KCB 없음 - 건너뜀")

# 관리비: 호별 ㎡당 월평균 관리비 (큰 파일이라 나눠 읽기)
try:
    parts = []
    for ch in common.read_csv("fee", chunksize=500000):
        ch = common.rename(ch, config.FEE_COLS)
        ch["당월관리비"] = pd.to_numeric(ch["당월관리비"], errors="coerce")
        parts.append(ch.groupby("호관리번호")["당월관리비"].agg(["sum", "count"]))
    fee = pd.concat(parts).groupby(level=0).sum()
    unit = unit.merge((fee["sum"] / fee["count"]).rename("월평균관리비"), left_on="호관리번호", right_index=True, how="left")
    unit["㎡당관리비"] = unit["월평균관리비"] / unit["전용면적"]
except FileNotFoundError:
    print("관리비 없음 - 건너뜀")

unit["장기공실호"] = ((unit["최장공실개월"] >= 12) | (unit["현재공실개월"] >= 12)).astype(int)
num = [c for c in ["전용면적", "방수", "준공연차", "임대보증금", "임대료", "㎡당임대료", "초기미임대개월",
                   "중도해약수", "저소득비율", "2030비율", "㎡당관리비"] if c in unit.columns]
cat = ["시도", "공급유형", "주택유형", "난방방식"]
X = pd.get_dummies(unit[num + cat], columns=cat, dummy_na=True).astype(float)
X = X.fillna(X.median())
y = unit["장기공실호"]
print("장기공실 호 비율: %.1f%%" % (100 * y.mean()))

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
res = []
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
res.append({"모델": "로지스틱회귀", "AUC": roc_auc_score(yte, lr.predict_proba(Xte)[:, 1])})
gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
res.append({"모델": "그래디언트부스팅", "AUC": roc_auc_score(yte, gb.predict_proba(Xte)[:, 1])})
common.save(pd.DataFrame(res).round(3), "04_모델성능.csv")

pi = permutation_importance(gb, Xte, yte, n_repeats=5, random_state=0, scoring="roc_auc")
imp = pd.DataFrame({"변수": X.columns, "중요도": pi.importances_mean}).sort_values("중요도", ascending=False)
common.save(imp.round(4), "04_변수중요도.csv")
coef = pd.DataFrame({"변수": X.columns, "계수(표준화)": lr[-1].coef_[0]}).sort_values("계수(표준화)")
common.save(coef.round(4), "04_로지스틱_계수.csv")

# 시군구별 예측 위험(평균) - 집계만 반출
unit["예측위험"] = gb.predict_proba(X)[:, 1]
r = unit.groupby(["시도", "시군구"]).agg(호수=("호관리번호", "size"), 평균예측위험=("예측위험", "mean"),
                                       실제장기공실비율=("장기공실호", "mean")).reset_index()
common.save(common.mask_small(r.round(3)), "04_시군구_예측위험.csv")

print(pd.DataFrame(res))


In [ ]:
%%writefile s05charts.py
"""[집에서 실행] 반출한 CSV로 기획서용 그래프 만들기.

센터 반출이 CSV만 가능하므로 그래프는 밖에서 그린다.
사용: 반출받은 CSV를 output/export/ 에 넣고 python s05charts.py
결과: output/charts/*.png
"""
import os
import pandas as pd
import common
import config

EXP = os.path.join(config.OUT_DIR, "export")
CH = common.ensure_out("charts")
plt = common.setup_korean_font()
PURPLE, ORANGE = "#4b3a9a", "#e8612c"


def load(name):
    path = os.path.join(EXP, name)
    if not os.path.exists(path):
        print("없음(건너뜀):", name)
        return None
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["호수"] = pd.to_numeric(df["호수"], errors="coerce") if "호수" in df.columns else None
    return df


def bar(df, x, y, title, fname, horizontal=False, color=PURPLE, top=None):
    d = df.dropna(subset=[y])
    if top:
        d = d.nlargest(top, y)
    fig, ax = plt.subplots(figsize=(7, 4.5 if horizontal else 4))
    if horizontal:
        ax.barh(d[x].astype(str), d[y], color=color); ax.invert_yaxis(); ax.set_xlabel(y)
    else:
        ax.bar(d[x].astype(str), d[y], color=color); ax.set_ylabel(y); plt.xticks(rotation=30, ha="right")
    ax.set_title(title)
    plt.tight_layout(); plt.savefig(os.path.join(CH, fname), dpi=200); plt.close()
    print("그래프:", fname)


sgg = load("03_시군구별.csv")
if sgg is not None:
    sgg["지역"] = sgg["시도"].str[:2] + " " + sgg["시군구"]
    bar(sgg, "지역", "현재공실률(%)", "현재 공실률 상위 시군구", "01_시군구_공실률_상위.png", True, ORANGE, top=15)
    bar(sgg, "지역", "공실_12개월이상비율", "12개월 이상 장기공실 비율 상위 시군구", "02_시군구_장기공실_상위.png", True, ORANGE, top=15)
for f, col, title, out in [
    ("03_시도별.csv", "시도", "시도별 현재 공실률", "03_시도별_공실률.png"),
    ("03_면적구간별.csv", "면적구간", "면적별 현재 공실률", "04_면적별_공실률.png"),
    ("03_공급유형별.csv", "공급유형", "공급유형별 현재 공실률", "05_공급유형별_공실률.png"),
    ("03_준공연차별.csv", "준공연차구간", "준공연차별 현재 공실률", "06_준공연차별_공실률.png"),
    ("03_임대료분위별.csv", "임대료분위_시군구내", "시군구 내 임대료 수준별 현재 공실률", "07_임대료분위별_공실률.png"),
    ("03_방수별.csv", "방수", "방 수별 현재 공실률", "08_방수별_공실률.png"),
]:
    d = load(f)
    if d is not None:
        bar(d.sort_values(col), col, "현재공실률(%)", title, out)

d = load("03_공급유형x면적.csv")
if d is not None:
    p = d.pivot_table(index="공급유형", columns="면적구간", values="현재공실률(%)")
    p = p[[c for c in config.AREA_LABELS if c in p.columns]]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    im = ax.imshow(p.values, cmap="Oranges", aspect="auto")
    ax.set_xticks(range(len(p.columns))); ax.set_xticklabels(p.columns)
    ax.set_yticks(range(len(p.index))); ax.set_yticklabels(p.index)
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            v = p.values[i, j]
            ax.text(j, i, "" if pd.isna(v) else "%.1f" % v, ha="center", va="center", fontsize=8)
    fig.colorbar(im, label="현재 공실률(%)"); ax.set_title("공급유형 × 면적 현재 공실률")
    plt.tight_layout(); plt.savefig(os.path.join(CH, "09_공급유형x면적_히트맵.png"), dpi=200); plt.close()
    print("그래프: 09_공급유형x면적_히트맵.png")

d = load("03_연도별_공실발생.csv")
if d is not None:
    d = d.dropna(subset=["공실_중위개월"])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(d["공실시작연도"], d["공실_중위개월"], marker="o", color=PURPLE)
    ax.set_ylabel("공실 기간 중위값(개월)"); ax.set_title("공실 시작 연도별 공실 기간 추이")
    plt.tight_layout(); plt.savefig(os.path.join(CH, "10_연도별_공실기간.png"), dpi=200); plt.close()
    print("그래프: 10_연도별_공실기간.png")

d = load("04_변수중요도.csv")
if d is not None:
    bar(d.head(15), "변수", "중요도", "장기공실 예측 변수 중요도 (상위 15)", "11_변수중요도.png", True, PURPLE)
d = load("04_시군구_예측위험.csv")
if d is not None:
    d["지역"] = d["시도"].str[:2] + " " + d["시군구"]
    bar(d, "지역", "평균예측위험", "장기공실 예측 위험 상위 시군구", "12_시군구_예측위험_상위.png", True, ORANGE, top=15)
print("완료:", CH)


In [ ]:
%%writefile s06dashboard.py
"""[집에서 실행] 반출 CSV를 묶어 대시보드 HTML 한 파일 생성.

사용: python s06dashboard.py   → ../app/dashboard.html
반출 CSV가 output/export/ 에 있어야 한다. 실제 데이터로 바꾸면 다시 실행만 하면 된다.
"""
import json
import os
import pandas as pd
import config

EXP = os.path.join(config.OUT_DIR, "export")
APP = os.path.join(os.path.dirname(__file__), "..", "app")
FILES = {"sido": "03_시도별.csv", "sgg": "03_시군구별.csv", "area": "03_면적구간별.csv",
         "supply": "03_공급유형별.csv", "sxa": "03_공급유형x면적.csv", "built": "03_준공연차별.csv",
         "rent": "03_임대료분위별.csv", "rooms": "03_방수별.csv", "year": "03_연도별_공실발생.csv",
         "imp": "04_변수중요도.csv", "perf": "04_모델성능.csv", "risk": "04_시군구_예측위험.csv"}
data = {}
for k, f in FILES.items():
    p = os.path.join(EXP, f)
    if os.path.exists(p):
        df = pd.read_csv(p, encoding="utf-8-sig")
        data[k] = json.loads(df.to_json(orient="records", force_ascii=False))
    else:
        data[k] = []
        print("없음:", f)
is_fake = os.path.abspath(config.DATA_DIR).endswith("fake_data")
meta = {"fake": is_fake, "built": pd.Timestamp.now().strftime("%Y-%m-%d"), "min_cell": config.MIN_CELL}
_tp = os.path.join(APP, "template.html")
if not os.path.exists(_tp):  # 센터 반입본은 확장자 제한으로 template.txt
    _tp = os.path.join(os.path.dirname(os.path.abspath(__file__)), "template.txt")
tpl = open(_tp, encoding="utf-8").read()
os.makedirs(APP, exist_ok=True)
html = tpl.replace("/*__DATA__*/", "const DATA=" + json.dumps(data, ensure_ascii=False) + ";const META=" + json.dumps(meta, ensure_ascii=False) + ";")
out = os.path.join(APP, "dashboard.html")
open(out, "w", encoding="utf-8").write(html)
print("생성:", out, "(%.1f KB)" % (len(html.encode()) / 1024), "연습데이터" if is_fake else "실데이터")


In [ ]:
%%writefile runall.py
"""센터: python runall.py (CSV만 생성) / 집 연습: python runall.py --fake (가짜 데이터 + 그래프)
집에서 반출 CSV로 그래프만: python s05charts.py"""
import runpy
import sys

steps = ["s01profile.py", "s02vacancy.py", "s02bcomplex.py", "s03aggregate.py", "s04model.py"]
if "--charts" in sys.argv or "--fake" in sys.argv:
    steps = steps + ["s05charts.py", "s06dashboard.py"]
if "--fake" in sys.argv:
    steps = ["s00fake.py"] + steps[:-1] + ["s07match.py", steps[-1]]  # 07은 외부 공공 단지정보 필요(센터 밖)
for s in steps:
    print("\n==========", s, "==========")
    runpy.run_path(s, run_name="__main__")


In [ ]:
%%writefile template.txt
<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>빈집 레이더</title>
<style>
:root{--bg:#f6f5fb;--card:#fff;--ink:#222;--mute:#666;--line:#e3dff2;--p:#4b3a9a;--o:#e8612c;--p2:#eeeaf9}
@media (prefers-color-scheme:dark){:root:not([data-theme=light]){--bg:#15131f;--card:#1f1c2e;--ink:#eee;--mute:#aaa;--line:#332e4a;--p2:#2a2540}}
:root[data-theme=dark]{--bg:#15131f;--card:#1f1c2e;--ink:#eee;--mute:#aaa;--line:#332e4a;--p2:#2a2540}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:"Pretendard","Malgun Gothic","Apple SD Gothic Neo",sans-serif;font-size:14px}
header{background:var(--p);color:#fff;padding:14px 16px}header h1{margin:0;font-size:18px}header p{margin:4px 0 0;font-size:12px;opacity:.85}
.warn{background:#fff3cd;color:#6b4e00;padding:6px 16px;font-size:12px}
nav{display:flex;gap:4px;padding:8px 16px 0;overflow-x:auto;border-bottom:1px solid var(--line);background:var(--card)}
nav button{border:0;background:none;padding:10px 12px;font-size:14px;color:var(--mute);border-bottom:3px solid transparent;cursor:pointer;white-space:nowrap}
nav button.on{color:var(--p);border-color:var(--p);font-weight:700}
main{padding:16px;max-width:1200px;margin:0 auto}section{display:none}section.on{display:block}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(360px,1fr));gap:12px}
.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px}
.kpi{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin-bottom:12px}
.kpi .card b{display:block;font-size:24px;color:var(--p)}.kpi .card span{font-size:12px;color:var(--mute)}
h2{font-size:15px;margin:0 0 10px}h3{font-size:13px;margin:0 0 8px;color:var(--mute)}
.ch{width:100%;height:320px;display:block}.tall .ch{height:460px}.ch text{fill:var(--ink);font-size:11px}.ch .ax{stroke:var(--line)}.ch .tip{font-size:10px;fill:var(--mute)}
table{width:100%;border-collapse:collapse;font-size:12px;white-space:nowrap}th,td{padding:6px 8px;border-bottom:1px solid var(--line);text-align:right}
th:first-child,td:first-child,th:nth-child(2),td:nth-child(2){text-align:left}.heat th,.heat td{white-space:normal;font-size:11px}th{cursor:pointer;background:var(--p2);position:sticky;top:0}
.tbl{max-height:420px;overflow:auto}.ctrl{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:10px}
select,input{padding:6px 8px;border:1px solid var(--line);border-radius:6px;background:var(--card);color:var(--ink);font-size:13px}
.bar{height:8px;background:var(--o);border-radius:4px;display:inline-block;vertical-align:middle}
.heat td{text-align:center}.heat td.c{color:#fff;font-weight:600}
.prop .card{border-left:4px solid var(--o)}.prop b{display:block;margin-bottom:4px}.prop p{margin:0;font-size:13px;color:var(--mute)}
footer{color:var(--mute);font-size:12px;padding:16px;text-align:center}
</style></head><body>
<header><h1>빈집 레이더 — LH 임대주택 장기공실 진단·예측</h1><p>2026 데이터안심구역 경진대회 · 호별 계약 이력 기반 공실 분석 (호수 <span id=mc></span> 미만 칸은 비공개)</p></header>
<div class=warn id=warn hidden>⚠ 지금 보이는 숫자는 코드 검증용 <b>연습(가짜) 데이터</b>입니다. 실제 분석 결과가 아닙니다.</div>
<nav><button data-t=ov class=on>개요</button><button data-t=rg>지역 탐색</button><button data-t=fx>공실 요인</button><button data-t=pr>위험 예측</button><button data-t=pp>제안·방법</button></nav>
<main>
<section id=ov class=on>
 <div class=kpi id=kpis></div>
 <div class=grid><div class=card><h2>시도별 현재 공실률</h2><svg class=ch id=cSido></svg></div>
 <div class=card><h2>연도별 공실 기간 추이</h2><svg class=ch id=cYear></svg></div></div>
</section>
<section id=rg>
 <div class=card><div class=ctrl><select id=fSido><option value="">전체 시도</option></select><input id=fQ placeholder="시군구 검색"><select id=fSort><option value="현재공실률(%)">공실률순</option><option value="공실_12개월이상비율">장기공실 비율순</option><option value="호수">호수순</option><option value="현재공실_중위개월">현재공실 기간순</option></select></div>
 <div class=grid><div class="card tall"><h2>상위 15 시군구</h2><svg class=ch id=cSgg></svg></div>
 <div class=card><h2>시군구 표</h2><div class=tbl><table id=tSgg></table></div></div></div></div>
</section>
<section id=fx>
 <div class=grid>
 <div class=card><h2>면적별 공실률</h2><svg class=ch id=cArea></svg></div>
 <div class=card><h2>공급유형별 공실률</h2><svg class=ch id=cSupply></svg></div>
 <div class=card><h2>준공연차별 공실률</h2><svg class=ch id=cBuilt></svg></div>
 <div class=card><h2>시군구 내 임대료 수준별 공실률</h2><svg class=ch id=cRent></svg></div>
 <div class=card><h2>방 수별 공실률</h2><svg class=ch id=cRooms></svg></div>
 <div class=card><h2>공급유형 × 면적 공실률(%)</h2><div class=tbl><table id=tHeat class=heat></table></div></div>
 </div>
</section>
<section id=pr>
 <div class=grid><div class="card tall"><h2>장기공실 예측 변수 중요도</h2><svg class=ch id=cImp></svg></div>
 <div class=card><h2>모델 성능</h2><table id=tPerf></table><h3 style="margin-top:12px">예측 위험 상위 시군구</h3><div class=tbl><table id=tRisk></table></div></div></div>
</section>
<section id=pp>
 <div class="grid prop">
 <div class=card><b>1. 공실 위험 단지 선제 관리</b><p>예측 위험 상위 지역·유형에 모집 공고 우선 배치, 입주 조건·임대료 조정 검토</p></div>
 <div class=card><b>2. 소형 평형 재구성</b><p>장기공실이 집중된 소형(40㎡ 미만) 세대 통합 리모델링, 1~2인 가구 맞춤 설비 개선</p></div>
 <div class=card><b>3. 지역 수요 맞춤 공급</b><p>지역 소득·청년 비율과 공급유형의 불일치 지역에 평형·유형 조정, 신규 공급 입지 기준에 공실 위험 반영</p></div>
 <div class=card><b>4. 공실 조기경보</b><p>계약 종료 예정 호와 과거 공실 패턴으로 다음 분기 공실 발생 예측 → 사전 마케팅</p></div>
 </div>
 <div class=card style="margin-top:12px"><h2>데이터와 방법</h2>
 <p>② 상호제공: LH 건설임대 계약(2009.02~2026.07)·관리비(2023.01~), KCB 행정동 소득, SKT 유동인구(서울) · ① K-data 고유: 교통카드·소득분위(확인 중) · ③ 공공: 전월세 실거래가, 인구, 교통·시설 위치</p>
 <p><b>공실 정의</b>: 같은 호에서 계약 종료(해약 시 해약월) 다음 달부터 다음 계약 시작 전 달까지. 다음 계약이 없고 2026.07 이전 종료면 '공실 진행 중'. <b>장기공실 호</b>: 12개월 이상 공실 경험 또는 현재 12개월 이상 공실.</p>
 <p><b>모델</b>: 로지스틱 회귀·그래디언트 부스팅으로 장기공실 호 예측, 순열 중요도로 요인 해석. 모든 수치는 지역·유형 단위 집계이며 소수 칸은 비공개.</p></div>
</section>
</main>
<footer>빈집 레이더 · 생성일 <span id=built></span> · 데이터안심구역 반출 집계 기반</footer>
<script>
/*__DATA__*/
const $=s=>document.querySelector(s),P='#4b3a9a',O='#e8612c';
document.getElementById('mc').textContent=META.min_cell;$('#built').textContent=META.built;if(META.fake)$('#warn').hidden=false;
document.querySelectorAll('nav button').forEach(b=>b.onclick=()=>{document.querySelectorAll('nav button,section').forEach(e=>e.classList.remove('on'));b.classList.add('on');$('#'+b.dataset.t).classList.add('on');render();});
let render=()=>{};let sortKey='현재공실률(%)',sortDir=-1;
const num=v=>(v===null||v===undefined||v==='')?null:(typeof v==='number'?v:(isNaN(+v)?null:+v));
const ok=r=>num(r['호수'])!==null;
function draw(id,labels,vals,{h=false,color=P,label='',line=false,vals2=null,color2=O}={}){const el=$('#'+id);const W=el.clientWidth||600,H=el.clientHeight||320;el.setAttribute('viewBox',`0 0 ${W} ${H}`);el.innerHTML='';
const ns='http://www.w3.org/2000/svg',mk=(t,a,tx)=>{const e=document.createElementNS(ns,t);for(const k in a)e.setAttribute(k,a[k]);if(tx!=null)e.textContent=tx;el.appendChild(e);return e;};
const v=vals.map(x=>x==null?0:x),n=v.length;if(!n){mk('text',{x:W/2,y:H/2,'text-anchor':'middle'},'데이터 없음');return;}
const mx=Math.max(...v,...(vals2||[0]))*1.1||1;
if(h){const L=Math.min(150,W*0.35),T=8,bh=(H-T-8)/n;v.forEach((x,i)=>{const y=T+i*bh;mk('rect',{x:L,y:y+bh*0.15,width:(W-L-50)*x/mx,height:bh*0.7,rx:3,fill:color});mk('text',{x:L-6,y:y+bh*0.62,'text-anchor':'end'},String(labels[i]).slice(0,14));mk('text',{x:L+(W-L-50)*x/mx+4,y:y+bh*0.62,class:'tip'},fmtv(x));});return;}
const L=40,B=H-48,T=12,bw=(W-L-10)/n;mk('line',{x1:L,y1:B,x2:W-5,y2:B,class:'ax'});
for(let g=0;g<=4;g++){const y=B-(B-T)*g/4;mk('line',{x1:L,y1:y,x2:W-5,y2:y,class:'ax','stroke-dasharray':'2 3'});mk('text',{x:L-4,y:y+4,'text-anchor':'end',class:'tip'},fmtv(mx*g/4));}
labels.forEach((lb,i)=>{const t=mk('text',{x:L+bw*(i+.5),y:B+14,'text-anchor':n>8?'end':'middle'},String(lb).slice(0,10));if(n>8)t.setAttribute('transform',`rotate(-30 ${L+bw*(i+.5)} ${B+14})`);});
if(line){const pts=a=>a.map((x,i)=>`${L+bw*(i+.5)},${B-(B-T)*(x||0)/mx}`).join(' ');mk('polyline',{points:pts(v),fill:'none',stroke:color,'stroke-width':2});v.forEach((x,i)=>mk('circle',{cx:L+bw*(i+.5),cy:B-(B-T)*x/mx,r:3,fill:color}));
if(vals2){mk('polyline',{points:pts(vals2),fill:'none',stroke:color2,'stroke-width':2,'stroke-dasharray':'4 3'});mk('text',{x:W-8,y:T+10,'text-anchor':'end',class:'tip'},'실선: '+label+' / 점선: '+(arguments[3].label2||''));}}
else v.forEach((x,i)=>{const bh=(B-T)*x/mx;mk('rect',{x:L+bw*(i+.15),y:B-bh,width:bw*.7,height:bh,rx:3,fill:color});mk('text',{x:L+bw*(i+.5),y:B-bh-4,'text-anchor':'middle',class:'tip'},fmtv(x));});}
const fmtv=x=>x==null?'':(Math.abs(x)>=100?Math.round(x).toLocaleString():(+x).toFixed(Math.abs(x)<1?2:1));
function bar(id,labels,vals,o={}){draw(id,labels,vals,o);}
function simple(id,rows,key,color){rows=rows.filter(ok);bar(id,rows.map(r=>String(r[key])),rows.map(r=>num(r['현재공실률(%)'])),{color});}
function renderAll(){
(function(){const s=DATA.sido.filter(ok);const tot=s.reduce((a,r)=>a+num(r['호수']),0),vac=s.reduce((a,r)=>a+(num(r['현재공실호수'])||0),0);
const l12=DATA.area.filter(ok);const long=l12.length?Math.round(l12.reduce((a,r)=>a+(num(r['공실_12개월이상비율'])||0)*num(r['호수']),0)/l12.reduce((a,r)=>a+num(r['호수']),0)*10)/10:'-';
const top=[...DATA.sgg.filter(ok)].sort((a,b)=>num(b['현재공실률(%)'])-num(a['현재공실률(%)']))[0];
$('#kpis').innerHTML=[['분석 호수',tot.toLocaleString()+'호'],['현재 공실 호',vac.toLocaleString()+'호'],['현재 공실률',(100*vac/tot).toFixed(1)+'%'],['공실 중 12개월 이상',long+'%'],['공실률 최고 지역',top?top['시군구']+' '+num(top['현재공실률(%)']).toFixed(1)+'%':'-']].map(([k,v])=>`<div class=card><b>${v}</b><span>${k}</span></div>`).join('');
simple('cSido',[...s].sort((a,b)=>num(b['현재공실률(%)'])-num(a['현재공실률(%)'])),'시도',P);
const y=DATA.year.filter(ok);draw('cYear',y.map(r=>r['공실시작연도']),y.map(r=>num(r['공실_중위개월'])),{line:true,label:'공실 기간 중위값(개월)'});})();
// 지역
if(!$('#fSido').dataset.i){$('#fSido').dataset.i=1;[...new Set(DATA.sgg.map(r=>r['시도']))].forEach(s=>$('#fSido').insertAdjacentHTML('beforeend',`<option>${s}</option>`));}
const COLS=['시도','시군구','호수','현재공실률(%)','현재공실호수','현재공실_중위개월','공실_중위개월','공실_12개월이상비율','초기미임대_중위개월','중도해약_호당평균'];
function region(){const sd=$('#fSido').value,q=$('#fQ').value.trim();let rows=DATA.sgg.filter(r=>ok(r)&&(!sd||r['시도']===sd)&&(!q||String(r['시군구']).includes(q)));
rows.sort((a,b)=>((num(b[sortKey])??-1)-(num(a[sortKey])??-1))*-sortDir);const t=rows.slice(0,15);
bar('cSgg',t.map(r=>r['시도'].slice(0,2)+' '+r['시군구']),t.map(r=>num(r[sortKey])),{h:true,color:O,label:sortKey});
$('#tSgg').innerHTML='<tr>'+COLS.map(c=>`<th data-k="${c}">${c.replace('_',' ')}${c===sortKey?(sortDir<0?' ▼':' ▲'):''}</th>`).join('')+'</tr>'+rows.map(r=>'<tr>'+COLS.map(c=>`<td>${fmt(r[c])}</td>`).join('')+'</tr>').join('');
$('#tSgg').querySelectorAll('th').forEach(h=>h.onclick=()=>{if(sortKey===h.dataset.k)sortDir*=-1;else{sortKey=h.dataset.k;sortDir=-1;}$('#fSort').value=sortKey;region();});}
const fmt=v=>{const n=num(v);return n===null?(v??''):(Number.isInteger(n)?n.toLocaleString():n.toFixed(1));};
['fSido','fQ','fSort'].forEach(i=>$('#'+i).oninput=()=>{if(i==='fSort'){sortKey=$('#fSort').value;sortDir=-1;}region();});region();
// 요인
simple('cArea',DATA.area,'면적구간',P);simple('cSupply',DATA.supply,'공급유형',P);simple('cBuilt',DATA.built,'준공연차구간',P);simple('cRent',DATA.rent,'임대료분위_시군구내',P);simple('cRooms',DATA.rooms,'방수',P);
(function(){const rows=DATA.sxa.filter(ok);const areas=['30미만','30~40','40~50','50~60','60~85','85이상'].filter(a=>rows.some(r=>r['면적구간']===a));const sups=[...new Set(rows.map(r=>r['공급유형']))];
const mx=Math.max(...rows.map(r=>num(r['현재공실률(%)'])||0));
$('#tHeat').innerHTML='<tr><th>공급유형</th>'+areas.map(a=>`<th>${a}㎡</th>`).join('')+'</tr>'+sups.map(s=>'<tr><td style="text-align:left">'+s+'</td>'+areas.map(a=>{const r=rows.find(x=>x['공급유형']===s&&x['면적구간']===a);const v=r?num(r['현재공실률(%)']):null;if(v===null)return'<td>-</td>';const k=v/mx;return`<td class=c style="background:rgba(232,97,44,${(0.15+0.85*k).toFixed(2)})">${v.toFixed(1)}</td>`;}).join('')+'</tr>').join('');})();
// 예측
(function(){const imp=DATA.imp.slice(0,15);bar('cImp',imp.map(r=>r['변수']),imp.map(r=>num(r['중요도'])),{h:true,color:P,label:'순열 중요도'});
$('#tPerf').innerHTML='<tr><th style="text-align:left">모델</th><th>AUC</th></tr>'+DATA.perf.map(r=>`<tr><td>${r['모델']}</td><td>${fmt(r['AUC'])}</td></tr>`).join('');
const rk=DATA.risk.filter(ok).sort((a,b)=>num(b['평균예측위험'])-num(a['평균예측위험'])).slice(0,20);const mx=Math.max(...rk.map(r=>num(r['평균예측위험'])));
$('#tRisk').innerHTML='<tr><th>시도</th><th>시군구</th><th>호수</th><th>예측 위험</th><th>실제 장기공실</th></tr>'+rk.map(r=>`<tr><td>${r['시도']}</td><td>${r['시군구']}</td><td>${fmt(r['호수'])}</td><td><span class=bar style="width:${80*num(r['평균예측위험'])/mx}px"></span> ${(100*num(r['평균예측위험'])).toFixed(0)}%</td><td>${(100*num(r['실제장기공실비율'])).toFixed(0)}%</td></tr>`).join('');})();
}
render=renderAll;renderAll();window.addEventListener('resize',renderAll);
</script></body></html>


In [ ]:
%%writefile readme.txt
# LH 임대주택 장기 공실 분석 (2026 데이터안심구역 경진대회)

## 폴더 구조
- `code/` 센터에서 실행할 분석 코드
  - `s00fake.py` 연습용 가짜 데이터 생성 (실제 데이터 아님)
  - `s01profile.py` [1차 방문] 데이터 요약표 + 분석 PC 환경 정보
  - `s02vacancy.py` 호별 계약 이력으로 공실 구간 계산
  - `s03aggregate.py` 반출용 집계표·그래프 (소수 칸 자동 가림)
  - `s04model.py` KCB 소득·관리비 결합 + 장기공실 위험 모델
  - `s05charts.py` [집에서] 반출한 CSV로 기획서용 그래프 12종 생성
  - `s06dashboard.py` [집에서] 반출 CSV를 묶어 `app/dashboard.html` 생성 (인터넷 없이 열림)
- `app/` 웹 대시보드 '빈집 레이더' (`template.html` 원본, `dashboard.html` 생성물)
  - `config.py` 경로·컬럼명·가림 기준 설정, `common.py` 공통 함수
  - `runall.py` 전체 실행
- `docs/LH공실_분석설계서.pdf` 문제 정의, 가설, 분석 설계, 일정, 역할, 신청서 문구, 기획서 목차

## 집에서 연습 (가짜 데이터)
```
cd code
pip install pandas numpy matplotlib scikit-learn
python runall.py --fake
```
결과는 `output/export/`(반출용 CSV), `output/work/`(호 단위, 반출 금지), `output/charts/`(그래프)에 생긴다.

## 센터에서 실행
1. 코드 파일을 센터 반입 절차에 따라 반입한다.
2. 데이터 폴더 경로를 지정한다: `set LH_DATA_DIR=D:\data` (Windows) 또는 `config.py`의 `DATA_DIR` 수정
3. 1차 방문: `python s01profile.py` → `output/export/01_*` 반출 신청
4. 2차 이후: `python s02vacancy.py` → `s03aggregate.py` → `s04model.py`
5. **`output/export/`의 CSV만 반출 신청한다. `output/work/` 는 호 단위 자료라 반출하지 않는다.**
6. 집에서: 반출받은 CSV를 `output/export/`에 넣고 `python s05charts.py` → `output/charts/`에 그래프 생성

## 주의
- 반출은 CSV만 가능한 것으로 보고 센터 코드는 그래프를 만들지 않는다.
- 실제 파일 컬럼명이 정의서와 다르면 `config.py`의 컬럼 매핑을 고친다.
- 반출 가림 기준(`MIN_CELL`, 기본 10)은 센터 기준에 맞춘다.
- 센터에 sklearn이 없으면 04단계는 건너뛰고 01~03만으로 기획서를 쓸 수 있다.

## 단지 단위 분석 (02b, 07)
1. `s02bcomplex.py`: 법정동+준공년월+공급유형+주택유형이 같은 호를 하나의 **추정단지**로 묶어 단지별 공실률·장기공실 비율 산출 → `export/02b_추정단지별_공실.csv` (반출 대상)
2. `s07match.py [공공단지정보.csv]`: 반출한 추정단지를 공공 LH 단지정보(단지명·법정동·준공·세대수·좌표)와 점수 매칭 → `export/07_단지매칭.csv` (확실/불확실/후보없음)
3. 연습데이터는 같은 규칙으로 만들어 매칭이 100% 나오지만, 실제 데이터는 '불확실'이 생길 수 있으니 상위 단지는 직접 확인


## 실행
`DATA` 를 센터 데이터 폴더 경로로 바꾸세요. 1회차에는 `s01profile.py` 만, 이후엔 `runall.py --charts`.

In [ ]:
import os, subprocess
DATA = "/데이터폴더"   # ← 센터 안내 경로로 수정
os.environ["LH_DATA_DIR"] = DATA
print(subprocess.run(["python3", "s01profile.py"], capture_output=True, text=True))

In [ ]:
# 2회차 이후 전체 실행
# r = subprocess.run(["python3", "runall.py", "--charts"], capture_output=True, text=True); print(r.stdout[-5000:]); print(r.stderr[-3000:])